# Week 3 — Multilingual Quantum Kernel + KTA, XLMR
**Project**: Quantum Kernels — Multilingual + KTA-predicts-gap (the novel result)

**For each of 6 languages:**
1. Load Week 2 embeddings (`multiling_xlmr_<lang>_embeddings.npz`)
2. Compute IQP quantum kernel matrix (N=1000) — saved immediately per language
3. Quantum SVM macro-F1 -> quantum-vs-classical GAP
4. Quantum KTA (alignment of the quantum kernel with labels)

**Then the cross-language analysis:**
Scatter quantum KTA vs the quantum-classical gap across all 6 languages, fit a
trend, report Pearson + Spearman correlation. **If KTA predicts the gap, that is
the mechanism result** — the contribution that lifts this above a benchmark.

**Runtime:** ~83 min x 6 languages = ~8 hrs (each kernel saved immediately).
You can run languages across sessions — completed ones are skipped on re-run.

In [ ]:
!pip install pennylane pennylane-lightning scikit-learn numpy scipy --quiet
import pennylane as qml, numpy as np, time, json, pickle, os, shutil, warnings
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from scipy.stats import pearsonr, spearmanr
warnings.filterwarnings('ignore')
print(f"PennyLane {qml.__version__}")
DEVICE = qml.device('lightning.qubit', wires=8)

---
## Step 0: Storage + KTA helper

In [ ]:
TAG = 'multiling_xlmr'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")
def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")
def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)
def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"; fig.savefig(p, bbox_inches='tight', dpi=150); _persist_and_download(p)
def _find(fname):
    dp = os.path.join(SAVE_DIR, fname); return dp if os.path.exists(dp) else fname
print("Helpers ready (Drive + download).")

def centered_kernel_alignment(K, y):
    y = np.asarray(y); classes = np.unique(y)
    Y = np.zeros((len(y), len(y)))
    for c in classes:
        m = (y == c).astype(float); Y += np.outer(m, m)
    n = K.shape[0]; H = np.eye(n) - np.ones((n, n)) / n
    Kc = H @ K @ H; Yc = H @ Y @ H
    num = np.sum(Kc * Yc); den = np.sqrt(np.sum(Kc*Kc) * np.sum(Yc*Yc))
    return float(num / den) if den > 0 else 0.0
print('KTA helper ready.')

---
## Step 1: IQP kernel + config

In [ ]:
N_QUBITS = 8
LANGUAGES = ['english', 'spanish', 'german', 'arabic', 'chinese', 'hindi']

def iqp_feature_map(x, wires):
    n=len(wires)
    for w in wires: qml.Hadamard(wires=w)
    for i,w in enumerate(wires): qml.RZ(x[i],wires=w)
    for i in range(n-1):
        qml.CNOT(wires=[wires[i],wires[i+1]]); qml.RZ(x[i]*x[i+1],wires=wires[i+1]); qml.CNOT(wires=[wires[i],wires[i+1]])
@qml.qnode(DEVICE)
def kernel_circuit(x1,x2):
    iqp_feature_map(x1,wires=range(N_QUBITS)); qml.adjoint(iqp_feature_map)(x2,wires=range(N_QUBITS))
    return qml.probs(wires=range(N_QUBITS))
quantum_kernel=lambda a,b: float(kernel_circuit(a,b)[0])
def mf1(yt,yp): return float(f1_score(yt,yp,average='macro'))
print("IQP kernel ready.")

---
## Step 2: Per-language quantum runs

Loops over the 6 languages. For each, if its kernel matrix already exists in Drive
it's reused (so re-running after a crash skips finished languages). Otherwise it
computes and saves immediately.

In [ ]:
results_per_lang = {}

for lang in LANGUAGES:
    ltag = f'{TAG}_{lang}'
    emb_path = _find(f'{ltag}_embeddings.npz')
    if not os.path.exists(emb_path):
        print(f"[skip] {lang}: no Week 2 embeddings found ({ltag}_embeddings.npz)")
        continue

    print(f"\n{'='*52}\nLanguage: {lang}\n{'='*52}")
    data = np.load(emb_path)
    X_tr = data['X_tr_subset'].astype(np.float64); y_tr = data['y_tr_subset']
    X_te = data['X_te_subset'].astype(np.float64); y_te = data['y_te_subset']
    with open(_find(f'{ltag}_baseline_results.pkl'),'rb') as f:
        baselines = pickle.load(f)
    best_classical_f1 = max(v['f1'] for v in baselines.values())

    # reuse saved kernel matrix if present
    km_path = _find(f'{ltag}_kernel_matrices.npz')
    if os.path.exists(km_path):
        print("  reusing saved kernel matrix")
        km = np.load(km_path); K_train = km['K_train_quantum']; K_test = km['K_test_quantum']
    else:
        print(f"  computing quantum kernel (N={len(X_tr)}, ~83 min)...")
        t0=time.time()
        K_train = qml.kernels.kernel_matrix(X_tr, X_tr, quantum_kernel)
        K_test  = qml.kernels.kernel_matrix(X_te, X_tr, quantum_kernel)
        print(f"  done {(time.time()-t0)/60:.1f} min — saving immediately:")
        checkpoint(f'{ltag}_kernel_matrices.npz',
                   lambda p,a=K_train,b=K_test,c=X_tr,d=y_tr: np.savez(p,K_train_quantum=a,K_test_quantum=b,X_tr=c,y_tr=d))

    # quantum SVM + gap
    yp = SVC(kernel='precomputed', C=1.0).fit(K_train, y_tr).predict(K_test)
    q_f1 = mf1(y_te, yp); q_acc = float(accuracy_score(y_te, yp))
    gap = q_f1 - best_classical_f1     # quantum minus best classical (macro F1)

    # quantum KTA on the train kernel
    q_kta = centered_kernel_alignment(K_train, y_tr)

    results_per_lang[lang] = {
        'quantum_f1': q_f1, 'quantum_acc': q_acc,
        'best_classical_f1': best_classical_f1, 'gap': gap,
        'quantum_kta': q_kta, 'baselines': baselines
    }
    print(f"  quantum macro-F1={q_f1:.4f}  best classical={best_classical_f1:.4f}  GAP={gap:+.4f}")
    print(f"  quantum KTA={q_kta:.4f}")

    # checkpoint progress after each language
    checkpoint(f'{TAG}_kta_gap_progress.json', lambda p,r=results_per_lang: json.dump(r,open(p,'w'),indent=2))

print(f"\nCompleted languages: {list(results_per_lang.keys())}")

---
## Step 3: THE NOVEL RESULT — does KTA predict the gap across languages?

Scatter quantum KTA (x) vs quantum-classical gap (y) across languages, fit a line,
report Pearson and Spearman correlation. A strong positive correlation means KTA,
computable cheaply before the kernel run, predicts whether the quantum kernel helps
for a given language — a mechanism, not just a benchmark.

In [ ]:
langs = list(results_per_lang.keys())
ktas = np.array([results_per_lang[l]['quantum_kta'] for l in langs])
gaps = np.array([results_per_lang[l]['gap'] for l in langs])

print(f"{'Language':<12} {'QuantumKTA':>11} {'Gap(Q-C)':>10}")
print("-"*35)
for l in langs:
    print(f"{l:<12} {results_per_lang[l]['quantum_kta']:>11.4f} {results_per_lang[l]['gap']:>+10.4f}")

if len(langs) >= 3:
    pear_r, pear_p = pearsonr(ktas, gaps)
    spear_r, spear_p = spearmanr(ktas, gaps)
    print(f"\nPearson  r={pear_r:+.3f}  p={pear_p:.3f}")
    print(f"Spearman r={spear_r:+.3f}  p={spear_p:.3f}")

    fig, ax = plt.subplots(figsize=(7,6))
    ax.scatter(ktas, gaps, s=90, color='#D95F02', edgecolor='white', zorder=3)
    for l,xk,yg in zip(langs, ktas, gaps):
        ax.annotate(l, (xk,yg), textcoords="offset points", xytext=(7,4), fontsize=10)
    # trend line
    if len(langs) >= 2:
        z = np.polyfit(ktas, gaps, 1); xs = np.linspace(ktas.min(), ktas.max(), 50)
        ax.plot(xs, np.poly1d(z)(xs), '--', color='gray', alpha=0.7,
                label=f'trend (Pearson r={pear_r:+.2f})')
    ax.axhline(0, color='black', lw=0.8, alpha=0.5)
    ax.set_xlabel('Quantum Kernel-Target Alignment (KTA)', fontsize=12)
    ax.set_ylabel('Quantum − Classical macro-F1 gap', fontsize=12)
    ax.set_title('XLMR: does KTA predict the quantum-classical gap?\nacross languages', fontsize=12)
    ax.legend(); plt.tight_layout(); plt.show()
    save_figure(fig, f'{TAG}_KTA_predicts_gap')
else:
    print("Need >=3 languages for correlation. Run more languages first.")
    pear_r=pear_p=spear_r=spear_p=None

---
## Step 4: Save the analysis

In [ ]:
analysis = {'encoder':'xlmr','languages':langs,
            'quantum_kta':{l:results_per_lang[l]['quantum_kta'] for l in langs},
            'gap':{l:results_per_lang[l]['gap'] for l in langs},
            'quantum_f1':{l:results_per_lang[l]['quantum_f1'] for l in langs},
            'best_classical_f1':{l:results_per_lang[l]['best_classical_f1'] for l in langs},
            'pearson_r':pear_r,'pearson_p':pear_p,'spearman_r':spear_r,'spearman_p':spear_p}
checkpoint(f'{TAG}_kta_analysis.json', lambda p: json.dump(analysis,open(p,'w'),indent=2))
print(f"\n=== WEEK 3 + KTA DONE — {TAG} ===")
if pear_r is not None:
    verdict = "KTA PREDICTS the gap (strong)" if abs(pear_r)>0.6 else ("weak/no relationship" if abs(pear_r)<0.3 else "moderate relationship")
    print(f"Across {len(langs)} languages: Pearson r={pear_r:+.3f} -> {verdict}")

---
## Week 3 + KTA done — Multilingual XLMR

### Saved:
- `multiling_xlmr_<lang>_kernel_matrices.npz` (per language, saved immediately)
- `multiling_xlmr_KTA_predicts_gap.png/.pdf` — the scatter, the key figure
- `multiling_xlmr_kta_analysis.json` — correlations + all numbers

### Reading the result:
- **Strong positive Pearson r (>0.6):** KTA predicts the quantum-classical gap.
  This is the mechanism — you can measure KTA cheaply and forecast whether the
  quantum kernel will help for a language. Main-track-shaped finding.
- **Weak/no correlation (<0.3):** KTA does not predict the gap here. Still a clean
  honest result; the multilingual breadth stands, and you report KTA was not
  predictive (a real negative finding, not a failure).

### Run both encoders:
This is one of two — run the mBERT and XLM-R versions. If the KTA-gap relationship
holds on BOTH encoders, that is a much stronger claim than either alone.